# Notebook 02 - Geração de Espectrogramas

In [15]:
import os
import librosa
import numpy as np
import matplotlib.pyplot as plt

In [16]:
# Diretórios de entrada e saída
input_dir_a = '../../dataset/preprocessed_audio/Hall-Reverb/'  # Áudios de entrada
input_dir_b = '../../dataset/preprocessed_audio/Chorus/'  # Áudios de saída (target)
output_dir_a = '../../dataset/spectrograms/Hall-Reverb/'  # Espectrogramas de entrada
output_dir_b = '../../dataset/spectrograms/Chorus/'  # Espectrogramas de saída (target)

os.makedirs(output_dir_a, exist_ok=True)
os.makedirs(output_dir_b, exist_ok=True)

In [17]:
# Parâmetros
sample_rate = 22050  # Taxa de amostragem
max_duration = 5.0  # Duração máxima dos áudios (em segundos)
n_fft = 2048  # Tamanho da FFT
hop_length = 512  # Passo da FFT
n_mels = 216  # Número de mel-bins
expected_height = 1025  # Altura esperada para o espectrograma

In [18]:
# Função para gerar espectrograma mel
def generate_spectrogram(audio, sample_rate, n_fft, hop_length, n_mels, expected_height):
    # Gerar espectrograma mel
    S = librosa.feature.melspectrogram(y=audio, sr=sample_rate, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels)
    S_db = librosa.power_to_db(S, ref=np.max)

    # Ajustar a altura do espectrograma (cortar ou preencher com 0)
    if S_db.shape[0] > expected_height:
        S_db = S_db[:expected_height, :]  # Cortar se for maior
    elif S_db.shape[0] < expected_height:
        padding = expected_height - S_db.shape[0]
        S_db = np.pad(S_db, ((0, padding), (0, 0)), mode='constant')  # Preencher se for menor

    # Garantir que o espectrograma tenha a dimensão extra para o canal (1 canal para mono)
    S_db = np.expand_dims(S_db, axis=-1)  # Adiciona a dimensão do canal (tornando-a (1025, 216, 1))

    return S_db


In [20]:
# Função para processar os diretórios e gerar espectrogramas
def process_spectrograms(input_dir_a, input_dir_b, output_dir_a, output_dir_b, sample_rate, n_fft, hop_length, n_mels, expected_height):
    for root_a, _, files_a in os.walk(input_dir_a):
        # Determinar o caminho relativo e correspondente para input_dir_b
        relative_path = os.path.relpath(root_a, input_dir_a)
        corresponding_dir_b = os.path.join(input_dir_b, relative_path)
        
        # Saídas correspondentes
        output_subdir_a = os.path.join(output_dir_a, relative_path)
        output_subdir_b = os.path.join(output_dir_b, relative_path)

        # Garantir que o diretório correspondente exista
        if not os.path.exists(corresponding_dir_b):
            print(f"Subdiretório correspondente não encontrado: {corresponding_dir_b}")
            continue

        # Ordenar os arquivos para garantir o pareamento
        files_a = sorted(f for f in files_a if f.endswith('.wav'))
        files_b = sorted(f for f in os.listdir(corresponding_dir_b) if f.endswith('.wav'))

        if len(files_a) != len(files_b):
            print(f"Número de arquivos diferente em {relative_path}. Pulando esta pasta.")
            continue

        # Processar cada par de arquivos
        for file_a, file_b in zip(files_a, files_b):
            input_path_a = os.path.join(root_a, file_a)
            input_path_b = os.path.join(corresponding_dir_b, file_b)
            
            output_path_a = os.path.join(output_subdir_a, file_a.replace('.wav', '.npy'))
            output_path_b = os.path.join(output_subdir_b, file_b.replace('.wav', '.npy'))
            
            # Carregar o áudio
            audio_a, _ = librosa.load(input_path_a, sr=sample_rate)
            audio_b, _ = librosa.load(input_path_b, sr=sample_rate)

            # Gerar espectrogramas
            spectrogram_a = generate_spectrogram(audio_a, sample_rate, n_fft, hop_length, n_mels, expected_height)
            spectrogram_b = generate_spectrogram(audio_b, sample_rate, n_fft, hop_length, n_mels, expected_height)

            # Salvar espectrogramas como .npy
            os.makedirs(os.path.dirname(output_path_a), exist_ok=True)
            os.makedirs(os.path.dirname(output_path_b), exist_ok=True)
            np.save(output_path_a, spectrogram_a)
            np.save(output_path_b, spectrogram_b)

In [21]:
# %% Executar geração de espectrogramas
process_spectrograms(input_dir_a, input_dir_b, output_dir_a, output_dir_b, sample_rate, n_fft, hop_length, n_mels, expected_height)

print("Geração de espectrogramas concluída! Espectrogramas salvos.")

Geração de espectrogramas concluída! Espectrogramas salvos.
